# Food Waste Analytics Pipeline — Exploratory Analysis

**Dataset:** Global Food Wastage Dataset (2018–2024) — Kaggle, Atharva Soundankar
**Author:** Pietro Masiero

This notebook translates the SQL analysis in [`sql/02_analysis.sql`](../sql/02_analysis.sql)
into Python (pandas + matplotlib/seaborn), reproducing each query as a
visualization. It demonstrates the same analytical logic applied
professionally to the Waste Watch Brazil program at Sodexo, using public data.

**Sections:**
1. Top 20 countries by total waste
2. Waste per capita ranking
3. Global trend over time
4. Waste by food category
5. Year-over-year change by country
6. Brazil deep dive
7. Economic impact analysis


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 110

df = pd.read_csv("../data/global_food_wastage_dataset.csv")
df.head()


In [ ]:
df.info()
print(f"\nYears covered: {df['Year'].min()}–{df['Year'].max()}")
print(f"Countries: {df['Country'].nunique()}")
print(f"Food categories: {df['Food Category'].nunique()}")


## 1. Top 20 countries by total waste (latest year)

Equivalent to `sql/02_analysis.sql` — Query 1.


In [ ]:
latest_year = df["Year"].max()

top20_waste = (
    df[df["Year"] == latest_year]
    .groupby("Country", as_index=False)
    .agg(total_waste_tons=("Total Waste (Tons)", "sum"),
         total_economic_loss=("Economic Loss (Million $)", "sum"))
    .sort_values("total_waste_tons", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=top20_waste, y="Country", x="total_waste_tons", ax=ax)
ax.set_title(f"Top 20 Countries by Total Food Waste ({latest_year})")
ax.set_xlabel("Total Waste (Tons)")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

top20_waste


## 2. Waste per capita ranking (latest year)

Equivalent to Query 2.


In [ ]:
per_capita = (
    df[df["Year"] == latest_year]
    .groupby("Country", as_index=False)
    .agg(avg_waste_per_capita_kg=("Avg Waste per Capita (Kg)", "mean"),
         avg_household_waste_pct=("Household Waste (%)", "mean"))
    .sort_values("avg_waste_per_capita_kg", ascending=False)
    .head(20)
)
per_capita = per_capita.round(2)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=per_capita, y="Country", x="avg_waste_per_capita_kg",
            hue="Country", palette="rocket", legend=False, ax=ax)
ax.set_title(f"Waste per Capita Ranking ({latest_year})")
ax.set_xlabel("Avg Waste per Capita (Kg)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

per_capita


## 3. Global trend over time (2018–2024)

Equivalent to Query 3.


In [ ]:
trend = (
    df.groupby("Year", as_index=False)
    .agg(global_waste_tons=("Total Waste (Tons)", "sum"),
         global_economic_loss=("Economic Loss (Million $)", "sum"),
         countries_reporting=("Country", "nunique"),
         avg_waste_per_capita_kg=("Avg Waste per Capita (Kg)", "mean"))
    .sort_values("Year")
)
trend["avg_waste_per_capita_kg"] = trend["avg_waste_per_capita_kg"].round(2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.lineplot(data=trend, x="Year", y="global_waste_tons", marker="o", ax=axes[0])
axes[0].set_title("Global Waste Over Time")
axes[0].set_ylabel("Total Waste (Tons)")

sns.lineplot(data=trend, x="Year", y="global_economic_loss", marker="o",
             color="firebrick", ax=axes[1])
axes[1].set_title("Global Economic Loss Over Time")
axes[1].set_ylabel("Economic Loss (Million $)")

plt.tight_layout()
plt.show()

trend


## 4. Waste by food category (all years)

Equivalent to Query 4.


In [ ]:
by_category = (
    df.groupby("Food Category", as_index=False)
    .agg(total_waste_tons=("Total Waste (Tons)", "sum"),
         total_economic_loss=("Economic Loss (Million $)", "sum"),
         avg_waste_per_capita_kg=("Avg Waste per Capita (Kg)", "mean"))
    .sort_values("total_waste_tons", ascending=False)
)
by_category["pct_of_total_waste"] = (
    by_category["total_waste_tons"] / by_category["total_waste_tons"].sum() * 100
).round(2)
by_category["avg_waste_per_capita_kg"] = by_category["avg_waste_per_capita_kg"].round(2)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=by_category, y="Food Category", x="total_waste_tons",
            hue="Food Category", palette="mako", legend=False, ax=ax)
ax.set_title("Total Waste by Food Category (2018–2024)")
ax.set_xlabel("Total Waste (Tons)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

by_category


## 5. Year-over-year change by country

Equivalent to Query 5 (`LAG` window function → pandas `.shift()`).


In [ ]:
yearly_by_country = (
    df.groupby(["Country", "Year"], as_index=False)
    .agg(total_waste_tons=("Total Waste (Tons)", "sum"))
    .sort_values(["Country", "Year"])
)

yearly_by_country["prev_year_waste"] = (
    yearly_by_country.groupby("Country")["total_waste_tons"].shift(1)
)
yearly_by_country["pct_change_yoy"] = (
    (yearly_by_country["total_waste_tons"] - yearly_by_country["prev_year_waste"])
    / yearly_by_country["prev_year_waste"] * 100
).round(2)

yoy = yearly_by_country.dropna(subset=["pct_change_yoy"])

# Heatmap: countries x years, YoY % change
pivot = yoy.pivot(index="Country", columns="Year", values="pct_change_yoy")

fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(pivot, cmap="RdYlGn_r", center=0, annot=True, fmt=".0f",
            cbar_kws={"label": "YoY % change"}, ax=ax)
ax.set_title("Year-over-Year Waste Change by Country (%)")
plt.tight_layout()
plt.show()

yoy.sort_values(["Country", "Year"]).head(20)


## 6. Brazil deep dive

Equivalent to Query 6.


In [ ]:
brazil = (
    df[df["Country"] == "Brazil"]
    .sort_values(["Year"], ascending=False)
    [["Year", "Food Category", "Total Waste (Tons)", "Economic Loss (Million $)",
      "Avg Waste per Capita (Kg)", "Household Waste (%)"]]
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

brazil_by_year = brazil.groupby("Year", as_index=False)["Total Waste (Tons)"].sum()
sns.lineplot(data=brazil_by_year, x="Year", y="Total Waste (Tons)",
             marker="o", ax=axes[0], color="darkgreen")
axes[0].set_title("Brazil — Total Waste Over Time")

brazil_by_cat = (
    brazil.groupby("Food Category", as_index=False)["Total Waste (Tons)"]
    .sum().sort_values("Total Waste (Tons)", ascending=False)
)
sns.barplot(data=brazil_by_cat, y="Food Category", x="Total Waste (Tons)",
            hue="Food Category", palette="YlOrBr", legend=False, ax=axes[1])
axes[1].set_title("Brazil — Waste by Food Category")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

brazil.head(20)


## 7. Economic impact analysis

Equivalent to Query 7.


In [ ]:
econ = (
    df.groupby("Country", as_index=False)
    .agg(total_waste_tons=("Total Waste (Tons)", "sum"),
         total_economic_loss=("Economic Loss (Million $)", "sum"))
    .sort_values("total_economic_loss", ascending=False)
)
econ["economic_loss_per_ton_usd"] = (
    econ["total_economic_loss"] / econ["total_waste_tons"]
).round(4)
top20_econ = econ.head(20)

fig, ax = plt.subplots(figsize=(9, 8))
scatter = ax.scatter(
    top20_econ["total_waste_tons"],
    top20_econ["total_economic_loss"],
    s=top20_econ["economic_loss_per_ton_usd"] * 40,
    c=top20_econ["economic_loss_per_ton_usd"],
    cmap="plasma", alpha=0.8, edgecolors="black"
)
for _, row in top20_econ.iterrows():
    ax.annotate(row["Country"], (row["total_waste_tons"], row["total_economic_loss"]),
                fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("Total Waste (Tons)")
ax.set_ylabel("Total Economic Loss (Million $)")
ax.set_title("Economic Impact: Waste Volume vs. Loss (bubble = $/ton)")
plt.colorbar(scatter, label="Economic Loss per Ton (USD)")
plt.tight_layout()
plt.show()

top20_econ


## Key takeaways

- Fill in with your own narrative once you review the outputs above —
  e.g. which countries/categories drive the most waste, how Brazil compares
  globally, and which markets show the steepest YoY increases.
- This mirrors the same KPI logic (compliance, per-capita, reduction vs.
  baseline) used in the Waste Watch Brazil program, applied here to public
  data for portfolio purposes.

**Next steps (roadmap):**
- [ ] Migrate these transformations into dbt models (`staging` → `marts`)
- [ ] Add dbt tests (not null, accepted values, relationships)
- [ ] Optional: rebuild visuals in Plotly/Streamlit for an interactive app
